# Random Guide-Selection Strategies

Compares random strategies for selecting guide sets.

`driver.py` writes one `results.parquet` per run, where a run is a single
strategy/config (see `config.json`). Each entry in `RUNS` below is one run
directory, and strategies are overlaid across runs.

In [ ]:
import json
import math
from collections.abc import Sequence
from typing import Any, cast

import polars as pl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from helpers import (
    find_latest_run,
    load_driver_run,
    load_baseline,
    load_baseline_per_seed,
    compute_goal_reach,
    compute_goal_reach_matrix,
    compute_best_ratio_per_goal,
)
from plots import (
    finish_fig,
    plot_cost_boxplots,
    plot_normalized_vs_baseline,
    plot_best_ratio_per_goal,
    plot_reachability_summary,
    split_modes,
)

plt.rcParams.update({"figure.dpi": 240, "figure.figsize": (12, 8)})


# plt.xkcd()

In [ ]:
# Configuration

# Each run directory holds a single strategy/config (driver.py writes one
# results.parquet per run). List the runs to overlay as (display_name, run-dir
# pattern); strategy and k come from the data.
RUNS = [
    ("no-replacement-count", "run.5"),
    # ("with-replacement-count", ""),
    # ("no-replacement-naive", ""),
    # ("with-replacement-naive", ""),
    # ("smallest-overall", ""),
    # ("smallest-novel", ""),
]

# smallest_* strategies emit a single k=1 point per goal (no k sweep); the rest
# sweep k. Used to pick marker style and skip per-k best/std overlays.
SINGLE_POINT_STRATEGIES = {"smallest_overall", "smallest_novel"}

# tab10 is a ListedColormap whose .colors is a sequence of RGBA tuples, but
# plt.cm is typed as the base Colormap and .colors as a broad ArrayLike, so
# cast to the sequence-of-colors shape the plotting helpers expect.
PALETTE = cast(Sequence[Any], cast(ListedColormap, plt.cm.tab10).colors)
MARKER_CYCLE = ["o", "s", "D", "^", "v", "P", "X", "*"]

# Metrics carried through the cost/best-ratio plots below. `nodes_per_class` is
# dropped for now: it isn't informative here, and if wanted it belongs in its
# own plot rather than mixed in with the cost metrics. The derived column and
# summary-table aggregation still compute it, so re-adding it is just a matter
# of listing it here again.
METRICS = ["iters", "nodes", "classes", "total_time"]

METRIC_BEST_AGG = {
    "iters": "min",
    "nodes": "min",
    "classes": "min",
    "nodes_per_class": "max",
    "total_time": "min",
}


def mode_style(i, is_single_point=False):
    return {
        "color": PALETTE[i % len(PALETTE)],
        "marker": "x" if is_single_point else MARKER_CYCLE[i % len(MARKER_CYCLE)],
        "linestyle": "none" if is_single_point else "-",
        "markersize": 10 if is_single_point else 6,
        "linewidth": 0 if is_single_point else 1.5,
        "zorder": 5 if is_single_point else 2,
    }

In [ ]:
# ── Load & parse ──────────────────────────────────────────────────────
# One run == one strategy. We overlay runs as "modes", keeping the old
# dataset_frames[ds_name][mode_name] -> DataFrame shape (single implicit
# dataset) so the downstream plotting cells stay unchanged.
DS_NAME = "random-strategies"
ds_names = [DS_NAME]

# Built from RUNS: (mode_name, run_dir, is_single_point). The middle element is
# the run directory.
SAMPLING_MODES = []
dataset_frames = {DS_NAME: {}}
# DS_NAME -> (n_goals, n_trials) for annotating the multi-modes
dataset_n = {}

mode_dfs = dataset_frames[DS_NAME]
ref = None
for mode_name, pattern in RUNS:
    run_dir = find_latest_run(pattern)
    config = json.loads((run_dir / "config.json").read_text())
    strategy = config["strategy"]
    is_single = strategy in SINGLE_POINT_STRATEGIES

    df = load_driver_run(run_dir)
    # Drop empty-pool legs (k=0, nothing to sample): not a real guide-set size.
    df = df.filter(pl.col("k") > 0)
    df = df.with_columns((pl.col("nodes") / pl.col("classes")).alias("nodes_per_class"))
    mode_dfs[mode_name] = df
    SAMPLING_MODES.append((mode_name, run_dir, is_single))

    n_reached = df["reached"].sum()
    print(
        f"{mode_name}: {run_dir.name} strategy={strategy}: {len(df)} rows, "
        f"k={sorted(df['k'].unique().to_list())}, {n_reached} reached"
    )

    # driver.py early-stops a pair on first reach, so the row count per (goal, k)
    # varies. Expose (n_goals, configured attempts) for the plot annotations
    # rather than asserting a constant trial count.
    if not is_single:
        n_goals = df["goal"].n_unique()
        n_trials = config["attempts"]
        if ref is None:
            ref = (n_goals, n_trials)

# There is always at least one multi-point run, so ref is set.
assert ref is not None, "no multi-point run found to derive (n_goals, n_trials)"
dataset_n[DS_NAME] = ref

# Union of k values across all multi-point runs (single-point runs only have k=1).
k_values = sorted(
    {
        k
        for mode_name, _run_dir, is_single in SAMPLING_MODES
        if not is_single
        for k in mode_dfs[mode_name]["k"].unique().to_list()
    }
)

print(f"\nk values: {k_values}")
print(f"(n_goals, n_trials) [attempts] for multi-point modes: {dataset_n}")

In [ ]:
# ── Baselines (full eqsat, no guides) ────────────────────────────────────
# Each run's config.json points at its seed-terms folder, whose terms.json
# carries a per-seed `goal_egraph` block (full eqsat on the seed with no
# guides). `load_baseline` averages those over the Ok seeds. The overlaid runs
# share a seed set, so we average their per-run baselines into one dataset
# baseline (mode-independent unguided cost).
#
# We also keep the per-seed baselines (`per_seed_baselines[mode]`): every goal
# under a seed shares that seed's unguided cost, so the normalized plot below
# pairs each guided leg with its own seed's baseline before aggregating. A
# single scalar line instead would conflate "guides helped" with "which seeds
# were in the mix".
baselines = {}
per_seed_baselines = {}
for ds_name in ds_names:
    per_mode = [load_baseline(run_dir) for _mode_name, run_dir, _is_single in SAMPLING_MODES]
    metrics = per_mode[0].keys()
    baselines[ds_name] = {m: sum(b[m] for b in per_mode) / len(per_mode) for m in metrics}
for mode_name, run_dir, _is_single in SAMPLING_MODES:
    per_seed_baselines[mode_name] = load_baseline_per_seed(run_dir)

print("Baselines (full eqsat, no guides):")
for ds_name, vals in baselines.items():
    print(f"  {ds_name}: nodes={vals['nodes']:.1f}, time={vals['total_time']:.6f}s")

## Cost relative to per-seed baseline

The baseline is per-seed (each seed's full unguided eqsat), and every seed
spawns many goals. A single flat baseline averaged over all seeds would give a
cheap seed's goals and an expensive seed's goals the same reference, so a curve
dipping below the line would partly reflect which seeds are in the mix rather
than whether guides helped. We don't plot that.

Instead we divide each reached leg by its own seed's unguided cost, so the
reference is a horizontal line at `ratio = 1.0` (guides break even; below 1 is
cheaper than unguided). Per k we show the median ratio with a 25th-75th
percentile band (ratios are heavy-tailed, so the median suits them better than
a mean), and faint per-seed median curves behind it show the seed-to-seed
spread.

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    plot_normalized_vs_baseline(
        mode_dfs,
        SAMPLING_MODES,
        per_seed_baselines,
        ds_name,
        k_values,
        METRICS,
        mode_style,
        show_per_seed=True,
        n_goals=n_goals,
        n_trials=n_trials,
    )

## Reachability summary

Three views in one figure: leg-level reach rate vs k, goal coverage (fraction
of goals with at least one successful trial) vs k, and the CDF of per-goal
reachability by k.

In [ ]:
# Reachability summary: leg reach rate vs k, goal coverage vs k, and the CDF of
# per-goal reachability.
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    plot_reachability_summary(
        mode_dfs,
        SAMPLING_MODES,
        ds_name,
        k_values,
        PALETTE,
        mode_style,
        compute_goal_reach,
        n_goals=n_goals,
        n_trials=n_trials,
    )


## Cost distribution: iters and nodes by strategy at each k

Node count includes the guide egraph (i.e. `max(trial_nodes, guide_egraph_nodes)`).

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    # Tag each run with its display mode name so we group by mode (one strategy
    # per run means the raw `strategy` column can't distinguish overlaid runs).
    combined = pl.concat(
        [
            v.filter(pl.col("reached")).with_columns(pl.lit(name).alias("mode"))
            for name, v in mode_dfs.items()
        ]
    )
    mode_names = [m for m, _p, _s in SAMPLING_MODES]
    plot_cost_boxplots(
        combined,
        ds_name,
        [("iters", "iters"), ("nodes", "nodes (incl. guide egraph)")],
        k_values,
        mode_names,
        PALETTE,
        n_goals=n_goals,
        n_trials=n_trials,
        group_col="mode",
    )

## Summary table

In [ ]:
from IPython.display import display

for ds_name in ds_names:
    print(f"\n=== {ds_name} ===")
    combined = pl.concat(
        [v.with_columns(pl.lit(name).alias("mode")) for name, v in dataset_frames[ds_name].items()]
    )
    summary = (
        combined.group_by("mode", "k")
        .agg(
            pl.col("iters").mean().round(2).alias("avg_iters"),
            pl.col("nodes").mean().round(0).alias("avg_nodes"),
            pl.col("classes").mean().round(0).alias("avg_classes"),
            pl.col("nodes_per_class").mean().round(2).alias("avg_nodes_per_class"),
            pl.col("total_time").mean().round(4).alias("avg_time_s"),
            pl.col("reached").mean().round(3).alias("reached_rate"),
            (~pl.col("reached")).mean().round(3).alias("unreached_rate"),
            pl.col("reached").sum().alias("n_reached"),
            pl.len().alias("n_legs"),
        )
        .sort("k", "mode")
    )
    display(summary)

## Per-goal analysis

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    for mode_name, _run_dir, _is_single in SAMPLING_MODES:
        df = mode_dfs[mode_name]
        goal_reach = compute_goal_reach(df)
        print(
            f"{ds_name} / {mode_name}: {goal_reach.shape[0]} goal×k combinations, {goal_reach['goal'].n_unique()} unique goals"
        )

## Goal-level reachability heatmap

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    multi_modes, _ = split_modes(SAMPLING_MODES)
    n_modes = len(multi_modes)
    n_cols_hm = min(4, n_modes)
    n_rows_hm = math.ceil(n_modes / n_cols_hm)

    fig, axes = plt.subplots(
        n_rows_hm,
        n_cols_hm,
        figsize=(3 * n_cols_hm + 1, 5 * n_rows_hm),
        squeeze=False,
    )

    im = None
    rightmost_ax = None
    for si, (_i, mode_name, _file_prefix, _is_single) in enumerate(multi_modes):
        ax = axes[si // n_cols_hm][si % n_cols_hm]
        df = mode_dfs[mode_name]
        mode_k_values = sorted(df["k"].unique().to_list())

        _goal_order, matrix = compute_goal_reach_matrix(df, mode_k_values)

        im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
        ax.set_xticks(range(len(mode_k_values)))
        ax.set_xticklabels([str(k) for k in mode_k_values])
        ax.set_xlabel("k (number of guides)")
        ax.set_ylabel("Goal (sorted by avg reachability, hardest at top)")
        ax.set_yticks([])
        ax.set_title(mode_name)
        rightmost_ax = ax

    for si in range(n_modes, n_rows_hm * n_cols_hm):
        axes[si // n_cols_hm][si % n_cols_hm].set_visible(False)

    if im is not None and rightmost_ax is not None:
        cbar = fig.colorbar(
            im, ax=rightmost_ax, orientation="vertical", label="Reachability rate (%)"
        )
        cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
        cbar.set_ticklabels(["0%", "25%", "50%", "75%", "100%"])

    fig.subplots_adjust(left=0.05, top=0.88, bottom=0.1, wspace=0.3)

    finish_fig(fig, "Goal-level reachability heatmap", ds_name, n_goals=n_goals, n_trials=n_trials)

## Best-guide improvement over baseline (per goal)

For a given goal, of all attempts to reach it with a sampled guide, how much did
the best guide improve over the unguided baseline?

For every reached leg we form `ratio = metric / seed_baseline[metric]` (paired
per seed, so each leg is measured against its own seed's full unguided eqsat),
then per (goal, k) keep the minimum ratio: the single best guide for that goal
at that k. `ratio < 1` means the best guide is cheaper than unguided; the dashed
line at `ratio = 1.0` is break-even.

Across goals we summarise those per-goal bests with the median and a 25th-75th
percentile band (ratios are heavy-tailed, so the median suits them better than a
mean). This is per-goal (not per-seed) and unitless. Baselines come from each
run's seed-terms `goal_egraph` (see `load_baseline_per_seed`).

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    plot_best_ratio_per_goal(
        mode_dfs,
        SAMPLING_MODES,
        per_seed_baselines,
        ds_name,
        k_values,
        ["nodes", "total_time"],
        mode_style,
        compute_best_ratio_per_goal,
        show_per_goal=False,
        n_goals=n_goals,
        n_trials=n_trials,
    )